In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/anikannal/solar-power-generation-data/Plant_2_Generation_Data.csv
/kaggle/input/datasets/anikannal/solar-power-generation-data/Plant_2_Weather_Sensor_Data.csv
/kaggle/input/datasets/anikannal/solar-power-generation-data/Plant_1_Weather_Sensor_Data.csv
/kaggle/input/datasets/anikannal/solar-power-generation-data/Plant_1_Generation_Data.csv


**Ghi chú:** mô hình dự báo `capacity_factor` (= `power_kw` / `plant_peak_kw`, 0-1) thay vì `power_kw` thô, để service phóng ra từng hộ theo công suất lắp đặt riêng (`house_pv_kw`). `power_kw` chỉ dùng để tính `capacity_factor`.


**Đọc và ghép dữ liệu**

In [ ]:
import pandas as pd
import numpy as np

gen = pd.read_csv("/kaggle/input/datasets/anikannal/solar-power-generation-data/Plant_1_Generation_Data.csv")
weather = pd.read_csv("/kaggle/input/datasets/anikannal/solar-power-generation-data/Plant_1_Weather_Sensor_Data.csv")

gen_agg = gen.groupby("DATE_TIME").agg({
    "DC_POWER": "sum",
    "AC_POWER": "sum",
}).reset_index()

gen_agg["DATE_TIME"] = pd.to_datetime(gen_agg["DATE_TIME"], format="%d-%m-%Y %H:%M")
weather["DATE_TIME"] = pd.to_datetime(weather["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")

df = pd.merge(
    gen_agg,
    weather[["DATE_TIME", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]],
    on="DATE_TIME", how="inner"
)

df = df.rename(columns={"AC_POWER": "power_kw", "AMBIENT_TEMPERATURE": "temperature"})
df["power_kw"] = df["power_kw"] / 1000

df.head()

Số dòng sau khi ghép: (3157, 6)


,DATE_TIME,DC_POWER,power_kw,temperature,MODULE_TEMPERATURE,IRRADIATION
0,2020-06-01 00:00:00,0.0,0.0,23.128673,20.464305,0.0
1,2020-06-01 00:15:00,0.0,0.0,23.032562,20.341429,0.0
2,2020-06-01 00:30:00,0.0,0.0,22.967493,20.269493,0.0
3,2020-06-01 00:45:00,0.0,0.0,22.810594,20.198918,0.0
4,2020-06-01 01:00:00,0.0,0.0,22.611436,20.085866,0.0


**Làm sạch dữ liệu**

In [ ]:
def clean_data(df):
    df = df.copy()
    df = df.drop_duplicates(subset="DATE_TIME")
    df = df.sort_values("DATE_TIME").reset_index(drop=True)

    df["power_kw"] = df["power_kw"].interpolate(method="linear")
    df["temperature"] = df["temperature"].interpolate(method="linear")

    df = df[df["power_kw"] >= 0]
    return df

df_clean = clean_data(df)
print(f"Dữ liệu sau làm sạch: {len(df)} -> {len(df_clean)} dòng")

DATE_TIME             0
DC_POWER              0
power_kw              0
temperature           0
MODULE_TEMPERATURE    0
IRRADIATION           0
dtype: int64
Trước làm sạch: 3157 dòng | Sau: 3157 dòng


In [ ]:
df_clean["hour_check"] = df_clean["DATE_TIME"].dt.hour

suspicious = df_clean[
    ((df_clean["hour_check"] >= 19) | (df_clean["hour_check"] <= 5)) & (df_clean["power_kw"] > 0.5)
]
print(f"Số dòng bất thường (ban đêm mà có điện đáng kể): {len(suspicious)}")

noon_zero = df_clean[
    (df_clean["hour_check"].between(11, 13)) & (df_clean["power_kw"] == 0)
]
print(f"Số dòng bất thường (giữa trưa mà công suất = 0): {len(noon_zero)}")

df_clean = df_clean.drop(columns="hour_check")

Số dòng bất thường (ban đêm mà có điện đáng kể): 0
Số dòng bất thường (giữa trưa mà công suất = 0): 0


**Tạo đặc trưng**

In [ ]:
def create_features(df, plant_peak_kw):
    df = df.copy()
    df["hour"] = df["DATE_TIME"].dt.hour + df["DATE_TIME"].dt.minute / 60
    df["day_of_week"] = df["DATE_TIME"].dt.dayofweek

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    # capacity_factor = công suất / công suất đỉnh (0-1): không phụ thuộc quy mô nhà máy,
    # nên service phóng ra được cho từng hộ theo pv_kw riêng (house_kW = cf * pv_kw).
    # power_kw chỉ là dữ liệu gốc để tính cf, không đưa trực tiếp vào mô hình.
    df["capacity_factor"] = (df["power_kw"] / plant_peak_kw).clip(0, 1)

    steps_per_day = 24 * 60 // 15
    df["cf_lag_24h"] = df["capacity_factor"].shift(steps_per_day)

    df = df.dropna().reset_index(drop=True)
    return df

# công suất đỉnh lấy trên phần train (80% đầu) để không rò rỉ thông tin từ tập test
plant_peak_kw = float(df_clean["power_kw"].iloc[: int(len(df_clean) * 0.8)].max())
print("plant_peak_kw:", plant_peak_kw)

df_features = create_features(df_clean, plant_peak_kw)
df_features.head()

In [ ]:
feature_cols = ["temperature", "MODULE_TEMPERATURE", "IRRADIATION", "hour_sin", "hour_cos", "cf_lag_24h"]
target_col = "capacity_factor"

missing = [c for c in feature_cols if c not in df_features.columns]
assert not missing, f"Thiếu cột trong df_features: {missing}"

print(f"Features: {feature_cols} | Target: {target_col} | Số dòng: {len(df_features)}")

**Lưu**

In [ ]:
df_features.to_csv("/kaggle/working/solar_data_processed.csv", index=False)

Đã lưu file


**Giai đoạn 2****

**Tách train test**

In [8]:
def split_train_test(df, test_ratio=0.2):
    split_idx = int(len(df) * (1 - test_ratio))
    return df.iloc[:split_idx].reset_index(drop=True), df.iloc[split_idx:].reset_index(drop=True)

train_df, test_df = split_train_test(df_features)
print(f"Train: {len(train_df)} dòng | Test: {len(test_df)} dòng")

Train: 2448 dòng | Test: 613 dòng


**Mô hình baseline - Linear Regression**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

df_features["cf_next"] = df_features["capacity_factor"].shift(-1)
df_fair = df_features.dropna().reset_index(drop=True)

train_fair, test_fair = split_train_test(df_fair)

X_train_fair = train_fair[feature_cols]
y_train_fair = train_fair["cf_next"]
X_test_fair = test_fair[feature_cols]
y_test_fair = test_fair["cf_next"]

lr_fair = LinearRegression()
lr_fair.fit(X_train_fair, y_train_fair)
y_pred_fair = lr_fair.predict(X_test_fair)

mae_fair = mean_absolute_error(y_test_fair, y_pred_fair)
rmse_fair = np.sqrt(mean_squared_error(y_test_fair, y_pred_fair))
print(f"Linear Regression - MAE: {mae_fair:.4f} (cf) ~ {mae_fair * plant_peak_kw:.4f} kW | RMSE: {rmse_fair:.4f} (cf)")

**Chuẩn bị dữ liệu cho LSTM**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# chỉ scale đầu vào; target capacity_factor vốn đã nằm trong [0, 1] nên không cần scaler_y
scaler_X = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(train_df[feature_cols])
X_test_scaled = scaler_X.transform(test_df[feature_cols])

y_train = train_df[[target_col]].values
y_test = test_df[[target_col]].values

def create_sequences(X, y, window_size=96):
    X_seq, y_seq = [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i : i + window_size])
        y_seq.append(y[i + window_size])
    return np.array(X_seq), np.array(y_seq)

WINDOW = 96
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, WINDOW)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test, WINDOW)

print("Hình dạng dữ liệu train:", X_train_seq.shape)
print("Hình dạng dữ liệu test:", X_test_seq.shape)

xây dựng và huấn luyện LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, activation="tanh", input_shape=(WINDOW, len(feature_cols)), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")
model.summary()

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_test_seq, y_test_seq),
    epochs=30,
    batch_size=32,
    verbose=1
)

so sánh 2 mô hình

In [ ]:
y_pred_lstm = model.predict(X_test_seq).clip(0, 1)
y_test_actual = y_test_seq

mae_lstm = mean_absolute_error(y_test_actual, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test_actual, y_pred_lstm))

print("=== BẢNG SO SÁNH (đơn vị capacity_factor 0-1; nhân plant_peak_kw ra kW) ===")
print(f"Linear Regression - MAE: {mae_fair:.4f} ~ {mae_fair * plant_peak_kw:.3f} kW | RMSE: {rmse_fair:.4f}")
print(f"LSTM              - MAE: {mae_lstm:.4f} ~ {mae_lstm * plant_peak_kw:.3f} kW | RMSE: {rmse_lstm:.4f}")

In [ ]:
import joblib, json

model.save("/kaggle/working/lstm_solar_forecast.h5")
joblib.dump(scaler_X, "/kaggle/working/scaler_X.pkl")

model_config = {
    "feature_cols": feature_cols,
    "window": WINDOW,
    "slot_duration_sec": 900,
    "target": "capacity_factor (0-1, = power_kw / plant_peak_kw)",
    "plant_peak_kw": plant_peak_kw,
    "house_pv_kw": {"H01": 4.0, "H02": 3.5, "H03": 0.4, "H04": 0.3},
    "metrics": {"linear_regression_mae": float(mae_fair), "lstm_mae": float(mae_lstm)},
}
with open("/kaggle/working/model_config.json", "w") as f:
    json.dump(model_config, f, indent=2, ensure_ascii=False)
print("Đã lưu lstm_solar_forecast.h5, scaler_X.pkl, model_config.json")

In [ ]:
import json

sample_gen = df_features.tail(192).copy()
sample_gen["timestamp"] = sample_gen["DATE_TIME"].dt.strftime("%Y-%m-%dT%H:%M:%S")

gen_points = sample_gen[["timestamp", "temperature", "MODULE_TEMPERATURE", "IRRADIATION",
                         "power_kw", "capacity_factor"]].to_dict(orient="records")

with open("/kaggle/working/sample_generation_points.json", "w") as f:
    json.dump(gen_points, f)
print("Đã xuất", len(gen_points), "điểm generation")